In [1]:
import pandas as pd

file_path = (r"C:\Users\w7127363\Downloads\processed_olist.csv")

data = pd.read_csv(file_path)

print("Dataset shape:")
print(data.shape)

print("\nFirst 5 rows:")
display(data.head())

Dataset shape:
(99224, 24)

First 5 rows:


,order_status,target,number_of_items,total_price,total_freight,number_of_products,average_product_weight,average_product_length,average_product_height,average_product_width,...,customer_zip_code_prefix,customer_city,customer_state,number_of_sellers,seller_zip_code,seller_city,seller_state,delivery_days,delivery_delay_days,delivery_missing
0,delivered,1,1.0,29.99,8.72,1.0,500.0,19.0,8.0,13.0,...,3149,sao paulo,SP,1.0,9350.0,maua,SP,8.0,-8.0,0
1,delivered,1,1.0,118.70,22.76,1.0,400.0,19.0,13.0,19.0,...,47813,barreiras,BA,1.0,31570.0,belo horizonte,SP,13.0,-6.0,0
2,delivered,1,1.0,159.90,19.22,1.0,420.0,24.0,19.0,21.0,...,75265,vianopolis,GO,1.0,14840.0,guariba,SP,9.0,-18.0,0
3,delivered,1,1.0,45.00,27.20,1.0,450.0,30.0,10.0,20.0,...,59296,sao goncalo do amarante,RN,1.0,31842.0,belo horizonte,MG,13.0,-13.0,0
4,delivered,1,1.0,19.90,8.72,1.0,250.0,51.0,15.0,15.0,...,9195,santo andre,SP,1.0,8752.0,mogi das cruzes,SP,2.0,-10.0,0


In [2]:
X = data.drop(columns=["target"])
y = data["target"]

print("Features shape:")
print(X.shape)

print("\nTarget shape:")
print(y.shape)

print("\nTarget distribution:")
print(y.value_counts())

Features shape:
(99224, 23)

Target shape:
(99224,)

Target distribution:
target
1    84649
0    14575
Name: count, dtype: int64


In [3]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

print("\nNumber of numeric features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

Numeric features:
['number_of_items', 'total_price', 'total_freight', 'number_of_products', 'average_product_weight', 'average_product_length', 'average_product_height', 'average_product_width', 'average_product_photos', 'total_payment_value', 'payment_installments', 'number_of_payments', 'customer_zip_code_prefix', 'number_of_sellers', 'seller_zip_code', 'delivery_days', 'delivery_delay_days', 'delivery_missing']

Categorical features:
['order_status', 'customer_city', 'customer_state', 'seller_city', 'seller_state']

Number of numeric features: 18
Number of categorical features: 5


In [4]:
X["customer_zip_code_prefix"] = (
    X["customer_zip_code_prefix"].astype(str)
)

X["seller_zip_code"] = (
    X["seller_zip_code"].astype(str)
)

print("Updated data types:")
print(X[[
    "customer_zip_code_prefix",
    "seller_zip_code"
]].dtypes)

Updated data types:
customer_zip_code_prefix    object
seller_zip_code             object
dtype: object


In [5]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

print("\nNumber of numeric features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

Numeric features:
['number_of_items', 'total_price', 'total_freight', 'number_of_products', 'average_product_weight', 'average_product_length', 'average_product_height', 'average_product_width', 'average_product_photos', 'total_payment_value', 'payment_installments', 'number_of_payments', 'number_of_sellers', 'delivery_days', 'delivery_delay_days', 'delivery_missing']

Categorical features:
['order_status', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'seller_zip_code', 'seller_city', 'seller_state']

Number of numeric features: 16
Number of categorical features: 7


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training features: (79379, 23)
Testing features: (19845, 23)

Training target distribution:
target
1    67719
0    11660
Name: count, dtype: int64

Testing target distribution:
target
1    16930
0     2915
Name: count, dtype: int64


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [8]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:")
print(X_train_processed.shape)

print("\nProcessed testing shape:")
print(X_test_processed.shape)

Processed training shape:
(79379, 20955)

Processed testing shape:
(19845, 20955)


In [11]:
from scipy import sparse
import numpy as np

if sparse.issparse(X_train_processed):
    train_missing = np.isnan(X_train_processed.data).sum()
    test_missing = np.isnan(X_test_processed.data).sum()
else:
    train_missing = np.isnan(X_train_processed).sum()
    test_missing = np.isnan(X_test_processed).sum()

print("Missing values in processed training data:", train_missing)
print("Missing values in processed testing data:", test_missing)

Missing values in processed training data: 0
Missing values in processed testing data: 0


In [12]:
import os
import scipy.sparse

output_folder = r"C:\Users\w7127363\Downloads\feature_engineered"
os.makedirs(output_folder, exist_ok=True)

scipy.sparse.save_npz(
    os.path.join(output_folder, "X_train.npz"),
    X_train_processed
)

scipy.sparse.save_npz(
    os.path.join(output_folder, "X_test.npz"),
    X_test_processed
)

y_train.to_csv(
    os.path.join(output_folder, "y_train.csv"),
    index=False
)

y_test.to_csv(
    os.path.join(output_folder, "y_test.csv"),
    index=False
)

print("Feature-engineered datasets saved successfully.")
print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)

Feature-engineered datasets saved successfully.
Training shape: (79379, 20955)
Testing shape: (19845, 20955)
